# Patient-friendly radiology reports with an LLM

We will translate Norwegian radiology reports into patient-friendly language with a language model, and see how different instructions change what the model says.

## Before you start

1. Sign in with a Google account to run the notebook in Colab.
2. Run one cell at a time with `Shift + Enter`.
3. The first time, you get a warning that the notebook was not authored by Google. Click **Run anyway**.
4. If you get a `NameError`, Colab has lost the connection. Choose *Runtime → Run before* and run the cell again. With Claude you must paste the key again.
5. To keep your changes, save a copy (*File → Save a copy in Drive*).

## 1. Getting started

The notebook uses **Claude Sonnet 5.5** from Anthropic. It needs an API key, which you get from the instructor. If you have no key, choose **Gemini 2.5 Flash**, which is free in Colab but an older and weaker model. We use the model's default settings. The model does not answer exactly the same way every time, so running the same cell twice can give slightly different answers.

Choose a model in the `MODEL` field and run the cell below. With Claude, a field appears where you paste the key and press Enter. The key is hidden and is not stored in the notebook.

In [ ]:
#@title Choose a model and run this cell
MODEL = "Claude Sonnet 5.5"  # @param ["Claude Sonnet 5.5", "Gemini 2.5 Flash (free, no key)"]

import base64
import json
import re
import subprocess
import sys
from getpass import getpass

import requests
from IPython.display import Markdown, display

GEMINI_MODEL = "google/gemini-2.5-flash"
CLAUDE_MODEL = "claude-sonnet-5-5"
OPUS_MODEL = "claude-opus-5-5"  # only used for the image in section 9
# Anthropic's default effort level for each model
EFFORT = {CLAUDE_MODEL: "high", OPUS_MODEL: "medium"}
USE_CLAUDE = MODEL.startswith("Claude")
MODEL_NAME = "Claude Sonnet 5.5" if USE_CLAUDE else "Gemini 2.5 Flash"
claude = None


def connect_claude():
    """Install the Anthropic library, ask for the key and check that it works."""
    pip = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "anthropic>=0.125"],
        capture_output=True, text=True,
    )
    if pip.returncode != 0:
        print(pip.stderr[-1000:])
    import anthropic

    # Strip spaces and line breaks that often come along when pasting
    key = "".join(getpass("Paste the API key and press Enter: ").split())
    if not key:
        print("No key was pasted. Run the cell again, or choose Gemini.")
        return None
    client = anthropic.Anthropic(api_key=key, max_retries=4)
    try:
        # Counting tokens is free and checks both the key and the model
        client.messages.count_tokens(
            model=CLAUDE_MODEL, messages=[{"role": "user", "content": "test"}]
        )
    except anthropic.AuthenticationError:
        print("The key does not work. Check that you pasted the whole key, and run the cell again.")
        return None
    except anthropic.APIConnectionError:
        print("Could not reach Anthropic. Check your internet connection and run the cell again.")
        return None
    except anthropic.APIStatusError as e:
        print(f"Anthropic returned an error ({e.status_code}): {e.message}")
        return None
    return client


if USE_CLAUDE:
    claude = connect_claude()
else:
    from google.colab import ai


def ask(user_text, system=None):
    """Send user_text (and an optional system prompt) to the model and return the answer.

    Gemini in Colab takes one combined text, so there the system prompt goes in front.
    If something goes wrong, a short error message is shown and the answer is empty.
    """
    if USE_CLAUDE:
        return _ask_claude(user_text, system)
    return _ask_gemini(user_text, system)


def _ask_gemini(user_text, system):
    full = f"{system}\n\n{user_text}" if system else user_text
    try:
        return ai.generate_text(full, model_name=GEMINI_MODEL)
    except Exception as e:
        _problem(f"No answer from Gemini: {e}")
        return ""


def _ask_claude(user_text, system, model=CLAUDE_MODEL):
    import anthropic

    if claude is None:
        _problem("Claude is not ready. Run the model cell again and paste the key.")
        return ""
    extra = {"system": system} if system else {}
    try:
        r = claude.beta.messages.create(
            model=model,
            max_tokens=16000,
            output_config={"effort": EFFORT[model]},
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",
            messages=[{"role": "user", "content": user_text}],
            **extra,
        )
    except anthropic.AuthenticationError:
        _problem("The key no longer works. Ask the instructor.")
        return ""
    except anthropic.RateLimitError:
        _problem("Many people are querying the model at once. Wait half a minute and run the cell again.")
        return ""
    except anthropic.APIConnectionError:
        _problem("Could not reach Claude. Check your internet connection and try again.")
        return ""
    except anthropic.APIStatusError as e:
        _problem(f"Claude returned an error ({e.status_code}): {e.message}")
        return ""
    if r.stop_reason == "refusal":
        return "*The model declined to answer this request.*"
    return "".join(b.text for b in r.content if b.type == "text")


def ask_image(prompt, image_url, model="Claude Sonnet 5.5"):
    """Send an image and a question to Claude Sonnet 5.5 or Claude Opus 5.5."""
    if not USE_CLAUDE:
        _problem("Images only work with Claude in this notebook. See the text below the cell.")
        return ""
    try:
        r = requests.get(
            image_url,
            headers={"User-Agent": "patient-friendly-llm-workshop (https://github.com/skaliy/patient-friendly-llm)"},
            timeout=20,
        )
        r.raise_for_status()
    except requests.RequestException:
        _problem("Could not download the image. Try again in a moment.")
        return ""
    image = {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": r.headers.get("Content-Type", "image/png").split(";")[0],
            "data": base64.standard_b64encode(r.content).decode("utf-8"),
        },
    }
    model_id = OPUS_MODEL if "Opus" in model else CLAUDE_MODEL
    return _ask_claude([image, {"type": "text", "text": prompt}], None, model_id)


def show(text):
    """Display text as markdown."""
    display(Markdown(text))


def _problem(text):
    """Show a short error message instead of a long traceback."""
    show(f"**Error:** {text}")


def _json_list(text):
    """Find the first JSON list in a model answer. Returns None if there is none."""
    m = re.search(r"\[.*?\]", text, re.DOTALL)
    if not m:
        return None
    try:
        value = json.loads(m.group())
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, list) else None


def sml_search(term, limit=3):
    """Search Store medisinske leksikon (SML), exactly as the search on the website does.

    Returns a list of tuples (headword, definition, url),
    or an empty list if SML does not answer.
    """
    try:
        r = requests.get(
            "https://sml.snl.no/api/v1/search",
            params={"query": term, "limit": limit, "offset": 0},
            timeout=10,
        )
        r.raise_for_status()
        hits = r.json()
    except (requests.RequestException, ValueError):
        print(f"No answer from SML for '{term}'.")
        return []
    return [
        (h["headword"], h.get("first_sentences", ""), h["article_url"])
        for h in hits
    ]


def sml_lookup(term):
    """Find the best SML article for a term.

    Prefers an article with exactly the same headword, and follows short
    cross-references (e.g. "lungeemboli ... se lungeembolisme") to the
    main article. Returns (headword, definition, url) or None.
    """
    hits = sml_search(term, limit=5)
    if not hits:
        return None
    best = next((h for h in hits if h[0].lower() == term.lower()), hits[0])

    redirect = re.search(r"\bse ([^.,]+)\.?\s*$", best[1])
    if redirect and len(best[1]) < 150:
        target = redirect.group(1).strip()
        for h in sml_search(target, limit=5):
            if h[0].lower() == target.lower():
                return h
    return best


def extract_medical_terms(report):
    """Ask the model for medical terms a patient would not understand. The prompt is in Norwegian."""
    prompt = (
        "Trekk ut medisinske fagtermer fra denne radiologirapporten som en "
        "pasient sannsynligvis ikke forstår. Returner kun en JSON-liste med "
        'strenger, ingen forklaring. Eksempel: ["pneumothorax", "pleuravæske"]'
    )
    terms = _json_list(ask(report, system=prompt)) or []
    return [str(t) for t in terms]


def lookup_terms(terms):
    """Look up each term in SML. The same article is only included once."""
    articles = []
    seen = set()
    for term in terms:
        hit = sml_lookup(term)
        if not hit:
            continue
        headword, definition, url = hit
        if not definition or url in seen:
            continue
        seen.add(url)
        articles.append(
            {"term": term, "article": headword, "definition": definition, "url": url}
        )
    return articles


def show_articles(articles):
    """Print a list of SML articles: term, article and the start of the definition."""
    if not articles:
        print("  (none)")
    for a in articles:
        short = a["definition"]
        if len(short) > 160:
            short = short[:160].rsplit(" ", 1)[0] + " ..."
        print(f"• {a['term']} → {a['article']}")
        print(f"  {short}")
    print()


def filter_relevant(report, articles, instructions):
    """Let the model choose which SML articles are relevant for the report.

    If the answer cannot be parsed, all articles are kept.
    """
    if not articles:
        return []
    listing = "\n".join(
        f"{i}. {a['article']}: {a['definition']}" for i, a in enumerate(articles, 1)
    )
    numbers = _json_list(
        ask(f"Rapport:\n{report}\n\nArtikler:\n{listing}", system=instructions)
    )
    if numbers is None:
        print("Could not parse the model's answer. Keeping all articles.")
        return articles
    keep = {int(n) for n in numbers if str(n).strip().isdigit()}
    return [a for i, a in enumerate(articles, 1) if i in keep]


def translate_with_sml(report, system_prompt, articles):
    """Translate a report with SML definitions as extra context. The message is in Norwegian."""
    context = "\n".join(f"- **{a['article']}**: {a['definition']}" for a in articles)
    message = f"""Rapport:
{report}

SML-kontekst (for å forstå fagtermene):
{context}

Skriv en pasientvennlig oversettelse."""
    return ask(message, system=system_prompt)


if not USE_CLAUDE or claude is not None:
    print(f"Ready. The notebook uses {MODEL_NAME}.")

### Check that the model answers

If you get a short answer below, everything works.

In [ ]:
show(ask("Skriv dette med enkle ord: Ingen tegn til fraktur."))

## 2. What is a prompt?

A prompt is the message you send to the model. Today we use two kinds at the same time:

**System prompt (instructions)**
> Du oversetter radiologirapporter til pasientvennlig norsk. Bevar usikkerhet. Ikke legg til prognose ...

**User prompt (the content)**
> CT thorax med kontrast. Det påvises ...

The system prompt sets the frame: role, task and what the model should avoid. In chat apps like ChatGPT it is written by the vendor and invisible to you. The user prompt is the actual content, here the radiology report.

We will write three different system prompts and see what happens with the same report.

## 3. The material

Three short, **fictional** radiology reports in the same style as real ones.

In [ ]:
REPORT_1 = """CT thorax med intravenøs kontrast.
Det påvises en 18 mm rund fortetning i høyre overlapp, suspekt for malignitet.
Forstørrede lymfeknuter mediastinalt opp til 15 mm.
Ingen pleuravæske. Ingen tegn til lungeemboli."""

REPORT_2 = """MR caput.
Det ses en 8 mm hyperintens lesjon periventrikulært på FLAIR-sekvenser,
mest forenlig med kronisk iskemisk forandring.
Ingen patologisk kontrastopptak. Ingen tegn til akutt infarkt eller blødning."""

REPORT_3 = """Røntgen thorax.
Hjertestørrelse innen normalområde.
Det ses fortetning basalt i venstre lunge, kan passe med infiltrat.
Anbefaler klinisk korrelasjon. Ingen pneumothorax."""

REPORTS = {
    "Report 1 (CT thorax)": REPORT_1,
    "Report 2 (MR caput)": REPORT_2,
    "Report 3 (Chest X-ray)": REPORT_3,
}

for name, r in REPORTS.items():
    show(f"### {name}")
    print(r)

## 4. V1: Naive prompt

A naive approach with no rules and no examples.

In [ ]:
V1 = "Forklar denne radiologirapporten på et språk en pasient kan forstå."

show(ask(REPORT_1, system=V1))


### Discussion (5 min, in groups)

Run V1 on the other reports too (the cells below) and look for:

1. **False reassurance.** Does the model say something is "normal" or "harmless" when the report does not?
2. **Lost uncertainty.** Does the model drop words like *mistanke om*, *kan passe med* or *suspekt for*?
3. **Added information.** Does the model add prognosis, causes or symptoms that are not in the report?
4. **Missing recommendation.** Is "anbefaler klinisk korrelasjon" and the like ignored?

Note one concrete example per report. We will try to fix these in the next step.

In [ ]:
show("### Report 2 (MR caput)")
show(ask(REPORT_2, system=V1))

In [ ]:
show("### Report 3 (Chest X-ray)")
show(ask(REPORT_3, system=V1))

## 5. V2: Structured prompt with rules

A detailed system prompt with explicit rules. This is what most people mean by "prompt engineering".

In [ ]:
V2 = """Du oversetter radiologiske rapporter til pasientvennlig norsk (bokmål).

Regler:
- Skriv på et nivå en pasient uten medisinsk bakgrunn kan forstå.
- Bevar usikkerhet nøyaktig. Hvis rapporten sier "mistanke om", "kan passe med",
  eller "ingen tegn til", skal dette gjenspeiles i oversettelsen.
- Gjengi kun det rapporten eksplisitt nevner. Ikke legg til prognose, mulige
  årsaker, eller symptomer.
- Beskriv aldri funn som "normale" med mindre rapporten sier det eksplisitt.
- Plasser fagbegrepet i parentes etter den enkle forklaringen
  (f.eks. "væske i lungene (lungeødem)").
- Bruk en nøytral, saklig tone i tredjeperson. Ikke skriv "du" eller "din".
- Svar bare med selve oversettelsen, uten overskrift eller innledning."""

show(ask(REPORT_1, system=V2))


### Compare V1 and V2

What got better? What is still wrong?

A common pattern: V2 fixes the most obvious problems, but the model still misses the tone, forgets the parentheses, or is inconsistent from report to report.

## 6. V3: Prompt with examples (*few-shot*)

The model often does better when it sees examples of the desired answer (*few-shot*) than when it only gets rules (*zero-shot*).

V3 is V2 plus two worked examples.

In [ ]:
V3 = V2 + """

---
Eksempel 1:

Rapport:
CT abdomen/bekken med intravenøs kontrast. Blindtarmen fremstår utvidet med
tverrmål opp mot 13 mm. Det ses tydelig infiltrasjon i det periappendikulære
fettvevet, samt en appendikolitt basalt i lumen. Ingen tegn til fri væske.
Funnene er forenlige med akutt appendisitt.

Pasientvennlig oversettelse:
Undersøkelsen av buken viser tegn på akutt betennelse i blindtarmen (appendisitt). Den er forstørret,
og det er irritasjon i fettvevet rundt den. Det ses også en forkalkning
(appendikolitt) nederst i blindtarmen. Det er ikke tegn til væskeansamling i buken.

---
Eksempel 2:

Rapport:
MR columna lumbalis. I nivå L4/L5 påvises et venstresidig paramediant skiveprolaps
som komprimerer venstre L5-rot i recessen. Det er redusert skivehøyde i samme
nivå forenlig med degenerasjon.

Pasientvennlig oversettelse:
MR-bildene av korsryggen viser at skiven mellom 4. og 5. lendevirvel har presset
seg ut mot venstre side (prolaps). Prolapset trykker på en nerve i dette området.
Skiven i samme nivå har redusert høyde, noe som kan passe med slitasjeforandringer
(degenerasjon)."""

show(ask(REPORT_1, system=V3))


### Side by side: V1, V2, V3

Run the cell below to see all three in sequence on the same report.

In [ ]:
for name, prompt in [
    ("V1 (naive, one line)", V1),
    ("V2 (rules, no examples)", V2),
    ("V3 (rules and two examples)", V3),
]:
    show(f"## {name}")
    show(ask(REPORT_1, system=prompt))

### Discussion (5 min)

- How big is the difference between V1 and V2?
- How big is the difference between V2 and V3?
- What do the examples in V3 capture that the rules in V2 do not?

## 7. Looking up terms in SML

So far the model has only used what it was trained on. What happens if we also give it Norwegian medical terms with definitions from Store medisinske leksikon (SML), the Norwegian medical encyclopedia?

In [ ]:
#@title Show a diagram of the steps
# Draw the steps from report to final answer
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 2.6)
ax.axis("off")

boxes = [
    (0.2, "Report", "#E8F0FE", "black"),
    (2.2, "Model extracts\nmedical terms", "#FCE8E6", "black"),
    (4.2, "Look up\nin SML", "#FEF7E0", "black"),
    (6.2, "Model removes\nirrelevant hits", "#F3E8FD", "black"),
    (8.2, "Definitions\nadded to prompt", "#E6F4EA", "black"),
    (10.2, "Final\nanswer", "#4285F4", "white"),
]
w, h, y = 1.6, 1.2, 0.5

for x, text, color, text_color in boxes:
    ax.add_patch(
        patches.FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.05",
            linewidth=1.5, edgecolor="#5F6368", facecolor=color,
        )
    )
    ax.text(x + w / 2, y + h / 2, text,
            ha="center", va="center", fontsize=10, color=text_color)

# Arrows between the boxes
for (x1, *_), (x2, *_) in zip(boxes, boxes[1:]):
    ax.annotate("", xy=(x2, y + h / 2), xytext=(x1 + w, y + h / 2),
                arrowprops=dict(arrowstyle="->", lw=1.8, color="#5F6368"))

ax.text(6, 2.2,
        "The model makes two decisions along the way: which words need explaining, and which articles are relevant",
        ha="center", fontsize=10, style="italic", color="#5F6368")

plt.tight_layout()
plt.show()

### Step 1: Search SML

We search for two words from report 1 and fetch the two best hits for each. Note what comes up for *malignitet*.

In [ ]:
for term in ["lungeemboli", "malignitet"]:
    print(f"Search: {term}")
    for headword, definition, url in sml_search(term, limit=2):
        print(f"• {headword}")
        print(f"  Definition: {definition}")
        print(f"  URL: {url}")
    print()

### Step 2: Let the model pick out terms from a report

In [ ]:
terms = extract_medical_terms(REPORT_1)
print("The model thinks these terms need explaining:")
for t in terms:
    print(f"  • {t}")


### Step 3: Look up the terms in SML

Are all the articles about what the report describes?

In [ ]:
articles = lookup_terms(terms)
show_articles(articles)

### Step 4: Relevance filter

The search finds articles where the word is mentioned, not necessarily an article about the word itself. If we pass everything on, the model may include information about completely different conditions.

So we let the model sort the list first. The filter is a new system prompt, built the same way as the ones we wrote earlier.

In [ ]:
RELEVANCE_FILTER = """Du får en radiologirapport og en nummerert liste med artikler fra Store medisinske leksikon (SML).

Behold artikler som:
- forklarer et funn eller en tilstand som nevnes i rapporten, også når rapporten sier at den ikke er der (f.eks. "Ingen tegn til lungeemboli")
- beskriver anatomi der funnene er
- forklarer fagord pasienten trenger for å forstå rapporten

Fjern artikler om andre tilstander eller organer enn dem rapporten handler om.
Er du i tvil, behold artikkelen.

Returner kun en JSON-liste med numrene på artiklene du beholder, for eksempel [1, 3]."""

relevant = filter_relevant(REPORT_1, articles, RELEVANCE_FILTER)

print("Kept:")
show_articles(relevant)
print("Removed:")
show_articles([a for a in articles if a not in relevant])

### Step 5: Put it together

V3 three times on the same report: without SML, with all the hits from step 3, and with only the filtered articles from step 4.

In [ ]:
show("## V3 without SML")
show(ask(REPORT_1, system=V3))

show("## V3 with all SML hits (no filter)")
show(translate_with_sml(REPORT_1, V3, articles))

show("## V3 with filtered SML articles")
show(translate_with_sml(REPORT_1, V3, relevant))

### Discussion

- Does the model pick the right terms? Are any important ones missing?
- Did the filter remove something that should have stayed, or keep something that should have gone?
- Did the model include anything from the irrelevant articles in the unfiltered translation?
- Do the explanations get more precise, or just longer?

## 8. Bonus: Adapt to different readers

The same report, explained for different patients. Change the `RECIPIENT` field and run the cell.

In [ ]:
RECIPIENT = "en 70-åring uten medisinsk bakgrunn"  # @param {type:"string"}

V3_with_recipient = V3 + f"\n\nMottaker: {RECIPIENT}. Tilpass språknivå deretter."

show(ask(REPORT_1, system=V3_with_recipient))


Try for example:

- "et barn på 14 år"
- "en person med høyere utdanning, men ikke medisinsk bakgrunn"
- "en pasient som allerede vet at de har lungekreft og kun vil høre om utviklingen"

**The hard question:** Should a clinical tool make this adaptation automatically? Who chooses the recipient profile in a real system?

## 9. Closing: The model looks at an image

Language models can read images too. We send the X-ray below to the model twice: first with a simple question, then with a role and a clear goal, as we have practised today.

![Shoulder dislocation, X-ray](https://upload.wikimedia.org/wikipedia/commons/e/ea/Luxation_epaule.PNG)

*Image source: [Luxation epaule](https://commons.wikimedia.org/wiki/File:Luxation_epaule.PNG), Wikimedia Commons, MB, [CC BY-SA 2.5](https://creativecommons.org/licenses/by-sa/2.5/)*

In [ ]:
IMAGE_MODEL = "Claude Sonnet 5.5"  # @param ["Claude Sonnet 5.5", "Claude Opus 5.5"]
IMAGE = "https://upload.wikimedia.org/wikipedia/commons/e/ea/Luxation_epaule.PNG"

show("## Simple question")
show(ask_image("Hva ser du på dette bildet?", IMAGE, model=IMAGE_MODEL))

show("## With a role and a goal")
show(ask_image(
    "Du er radiolog og skal forklare funn på dette bildet til en pasient uten medisinsk bakgrunn. "
    "Skriv to-tre setninger på enkelt norsk.",
    IMAGE,
    model=IMAGE_MODEL,
))

What changed when the model got a role and a goal?

Now choose Claude Opus 5.5 in the `IMAGE_MODEL` field and run the cell again. Opus is larger and more expensive than Sonnet. Did the explanation get better?

The image shows a dislocated shoulder. Did the model find it?

If you are using Gemini, the cell above does not work. Paste the image into ChatGPT, Claude or the Gemini app and ask the same two questions there.